# 15 · scikit-learn — solutions (notebook exécuté)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cemah2/workbookIA/blob/main/chapitres/ch15_sklearn/05_solutions.ipynb)

Les réponses des exercices, exécutées. Les démarches détaillées (le *pourquoi*, les erreurs fréquentes, les variantes) sont dans `05_solutions.md`. Les cellules marquées `answer` enregistrent les réponses vérifiées par `wb.check` (`tools/build_answers.py`).

| Partie | ID | Titre | Type | ★ | ⏱️ |
|---|---|---|---|---|---|
| 0 | 15.Q1 à 15.Q12, 15.R1 à 15.R3, 15.1 à 15.5 | vérifier tes réponses courtes | 🧠 🔁 ✏️ 🧮 | ★ à ★★ | |
| A | 15.9 à 15.12 | l'interface d'un estimateur, ses trois sorties, les datasets et le découpage, le code de 2018 à faire tourner avec la version 1.6 (*prochaine session de génération*) | | | |
| B | 15.13 à 15.16 | clustering (KMeans, DBSCAN, hiérarchique), transformateurs et inverse_transform, PCA et blanchiment, ensembles sur cinq spirales (*prochaine session de génération*) | | | |
| C | 15.17 à 15.24 | validation croisée, recherches en grille et au hasard, frontières de décision, pipeline polynomial, lecture de cv_results_, la fuite du ch. 8 refaite avec un pipeline (*prochaine session de génération*) | | | |
| D | 15.25 à 15.31 | seuil de décision, classes déséquilibrées, six familles de modèles, ColumnTransformer sur les manchots bruts, ta librairie face à scikit-learn, sauvegarde, le défi (*prochaine session de génération*) | | | |

In [1]:
# ⚙️ SETUP: run this cell first (on Colab or on your computer)
FAST_MODE = True  # True = quick version (CPU, a few minutes); False = full version
SEED = 42

import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/cemah2/workbookIA.git"
DRIVE_DIR = Path("/content/drive/MyDrive/workbookIA")

try:
    import google.colab  # noqa: F401  (this module only exists on Colab)
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    subprocess.run(["git", "config", "--global", "--add", "safe.directory", str(DRIVE_DIR)])
    if not (DRIVE_DIR / ".git").exists():
        subprocess.run(["git", "clone", REPO_URL, str(DRIVE_DIR)], check=True)
    else:
        # Get the new chapters. Your own work only lives in mon_travail/, which Claude never
        # touches, so "rebase + autostash" keeps it safe (even with local commits).
        identity = subprocess.run(["git", "-C", str(DRIVE_DIR), "config", "user.email"],
                                  capture_output=True).returncode == 0
        who = [] if identity else ["-c", "user.name=Workbook", "-c", "user.email=workbook@localhost"]
        pull = subprocess.run(["git", *who, "-C", str(DRIVE_DIR), "pull", "--rebase", "--autostash"],
                              capture_output=True, text=True)
        conflicts = subprocess.run(["git", "-C", str(DRIVE_DIR), "diff", "--name-only", "--diff-filter=U"],
                                   capture_output=True, text=True).stdout.split()
        if conflicts:  # a file changed outside mon_travail/ AND by Claude: git leaves conflict markers in it
            print("⚠️ Tes modifications de " + ", ".join(conflicts) + " (hors de mon_travail/) entrent en conflit "
                  "avec la nouvelle version ; elles sont gardées dans `git stash`. Pour reprendre la version du "
                  "dépôt : git restore --source=HEAD --staged --worktree <fichier> (00_setup/COLAB.md, « Problèmes "
                  "fréquents »).")
        elif pull.returncode != 0:
            print("⚠️ git pull a échoué (voir 00_setup/COLAB.md, « Problèmes fréquents ») :\n" + pull.stderr)
    ROOT = DRIVE_DIR
else:
    ROOT = Path(os.environ.get("WB_ROOT", Path.cwd())).resolve()
    while not (ROOT / "src" / "wb").exists() and ROOT != ROOT.parent:
        ROOT = ROOT.parent
    if not (ROOT / "src" / "wb").exists():
        raise FileNotFoundError("Dépôt introuvable : ouvre ce notebook depuis le dossier du workbook.")

os.environ["WB_ROOT"] = str(ROOT)
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

import wb  # noqa: E402

cfg = wb.setup(seed=SEED, fast=FAST_MODE)
FAST_MODE = cfg.fast
mylearn = wb.load_mylearn("ref")  # reference implementation

🧪 Deep Learning Workbook : environnement
   Plateforme : Local (Linux x86_64, 2 CPU)
   Python 3.13.13 | numpy 2.1.3 | pandas 2.2.3 | matplotlib 3.10.0 | scikit-learn 1.6.1 | scipy 1.16.3 | torch 2.11.0+cpu | torchvision 0.26.0+cpu
   Device : cpu | Graine : 42 | FAST_MODE : True


## Objectifs et rappel express

- Utiliser l'interface commune de scikit-learn : hyperparamètres au constructeur, `fit` qui renvoie l'objet, attributs appris suffixés par `_`, `predict`, `decision_function`, `predict_proba`, `transform` et `inverse_transform`.
- Évaluer et régler un modèle : `cross_val_score`, `cross_validate`, `KFold`, `StratifiedKFold`, `GridSearchCV`, `RandomizedSearchCV`, et lire `cv_results_`.
- Enchaîner sans fuite : `Pipeline`, `make_pipeline`, `ColumnTransformer`, noms `étape__paramètre`.
- Moderniser du code ancien, choisir un seuil de décision, traiter des classes déséquilibrées, sauvegarder un pipeline avec ses versions.

**Rappel express.** $\hat p = \sigma(s) = 1/(1 + e^{-s})$ à deux classes, le softmax à $K$ classes ; nombre d'entraînements d'une grille : $\big(\prod_h n_h\big) \times k + 1$ ; probabilité qu'au moins un de $n_{\text{iter}}$ essais au hasard tombe dans le meilleur $q$ : $1 - (1 - q)^{n_{\text{iter}}}$ ; nombre de features polynomiales : $\binom{p + d}{d}$ ; min-max : $x' = (x - m)/(M - m)$. En scikit-learn : `sklearn.model_selection` (validation croisée, recherches, découpage), `sklearn.pipeline`, `sklearn.compose`, `sklearn.preprocessing`, `sklearn.inspection`, `sklearn.calibration` ; `joblib` pour la sauvegarde.

## Partie 0 · Vérifier tes réponses courtes (quiz, rappels, ✏️ 15.1 à 15.4 et 🧮 15.5)

Fais d'abord les quiz, les rappels et les exercices de `02_exercices.md` **sur papier**, dans ta copie de `06_mes_reponses.md`. Reporte ensuite chaque réponse courte ici : **la valeur** que tu as trouvée, pas l'expression Python, sinon tu ne vérifies rien. Un nombre : `42` ou `0.3` (en Python, le séparateur décimal est un **point** ; `0,3` sans guillemets serait un couple de deux nombres) ; un vrai ou faux : `True` ou `False` ; un choix : la lettre seule, entre guillemets (`"E"`) ; un nom : une chaîne (`"standardscaler"`) ; plusieurs nombres : une liste (`[4, 7]`). Arrondis comme l'énoncé le demande, et seulement à la fin du calcul. Les réponses pas encore remplies affichent ⏳. Les questions « dans ta copie », la réflexion (🗣️ 15.6, ⚖️ 15.7, 📄 15.8) et l'entretien se corrigent avec `05_solutions.md`.

In [2]:
# The short answers only use the numbers of their statements (02_exercices.md),
# and the defaults of the installed scikit-learn 1.6 where a question asks for one
import inspect
import math
from math import comb

import numpy as np
from scipy.special import expit, softmax
from sklearn.cluster import KMeans
from sklearn.model_selection import KFold, RandomizedSearchCV, train_test_split

# Q6, Q10, Q12, R3: defaults and sizes, read from scikit-learn itself
KMEANS_DEFAULT_K = inspect.signature(KMeans).parameters["n_clusters"].default
RANDOM_DEFAULT_ITER = inspect.signature(RandomizedSearchCV).parameters["n_iter"].default
_, TEST_Q12 = train_test_split(np.arange(125), test_size=0.3, random_state=0)
_, TEST_DEFAULT = train_test_split(np.arange(1000), random_state=0)
LARGEST_FOLD_R3 = max(len(val) for _, val in KFold(5).split(np.arange(1003)))


def fits(n_candidates, k, refit=True):
    """Trainings of a search: each candidate is cross-validated on k folds, plus the final refit."""
    return n_candidates * k + (1 if refit else 0)


# 15.1: the grids of the statement
GRID_151 = {"alpha": [0.01, 0.1, 1, 10], "solver": ["svd", "cholesky", "lsqr"]}
N_CAND_151 = math.prod(len(v) for v in GRID_151.values())
POLY_151 = 5 * 2 * 3                          # degree x interaction_only x alpha
LIST_151 = 3 * 1 + 3 * 2 * 1                  # two dictionaries, each with one kernel

# 15.2: scores of a binary logistic regression, then of a 3-class multinomial one
S_152 = np.array([2.0, -1.0, 0.0, 0.5, -3.2])
P_152 = expit(S_152)


def logit(p):
    return math.log(p / (1 - p))


PRED_152 = [int(s > 0) for s in S_152]        # scikit-learn: class 1 when the score is strictly positive
SOFT_152 = softmax(np.array([2.0, 1.0, -1.0]))

# 15.3: a MinMaxScaler learned on x in [-3, 5] and y in [10, 30], then the line y' = 0.6 x' + 0.25
X_MIN, X_MAX, Y_MIN, Y_MAX = -3.0, 5.0, 10.0, 30.0
A_153, B_153 = 0.6, 0.25


def x_scaled(x):
    return (x - X_MIN) / (X_MAX - X_MIN)


def y_back(y_scaled):
    return Y_MIN + (Y_MAX - Y_MIN) * y_scaled


def line_153(x):
    """The fitted line, in the original units."""
    return y_back(A_153 * x_scaled(x) + B_153)


SLOPE_153 = (line_153(X_MAX) - line_153(X_MIN)) / (X_MAX - X_MIN)

# 15.4: monomials of degree at most d in p variables


def n_poly(p, d, bias=True):
    return comb(p + d, d) - (0 if bias else 1)


SMALLEST_D_154 = next(d for d in range(1, 20) if n_poly(10, d) > 1000)

# 15.5: a 4-minute training, 5 hyperparameters, 5 folds
FITS_155 = 5 ** 5 * 5
DAYS_155 = FITS_155 * 4 / 60 / 24
HALVING_155 = sum(n * r for n, r in zip((81, 27, 9, 3, 1), (1 / 81, 3 / 81, 9 / 81, 27 / 81, 1)))
N_99_155 = math.ceil(math.log(0.01) / math.log(0.95))

**15.Q1 — Coder soi-même ou utiliser une bibliothèque ?**

In [3]:
wb.record("15.Q1a", False, mistakes={"relis le §15.1 : quel chemin le livre choisit-il pour ce chapitre ?": True})
wb.record("15.Q1b", "A", mistakes={"une bibliothèque sait-elle quelles données tu vas lui donner ?": "B",
          "peut-on choisir et régler un modèle sans savoir ce qu'il fait ?": "C",
          "les modèles de scikit-learn ont-ils des arguments à régler ? Relis le §15.4.1": "D"})
wb.record("15.Q1c", True, mistakes={"relis le §15.2 : sur quelles bibliothèques s'appuie scikit-learn ?": False})
wb.record("15.Q1d", "B", mistakes={"relis le §15.2 : à quoi scikit-learn n'est-il pas destiné ?": "A",
          "SciPy fournit-il des réseaux de neurones et le calcul sur GPU ?": "C",
          "seaborn sert-il à entraîner des modèles ?": "D"})

WB_ANSWER {"hash": "864089caf60c119af78b35f86901cae462e28c2083b05956da11600f803f36dc", "id": "15.Q1a", "kind": "bool", "mistakes": {"684826c91301e3ea10b72b0f9d59652d62e66be2f9e896950371febb60fc60b2": "relis le §15.1 : quel chemin le livre choisit-il pour ce chapitre ?"}}
📝 Ex 15.Q1a : réponse enregistrée (bool).
WB_ANSWER {"hash": "22d5e7b90c19c3345c757e06d91fe7bc61917dc72f4c02dac93187fbb0c69f6a", "id": "15.Q1b", "kind": "str", "mistakes": {"7ab9d4a24b3bd3730cbbdc0d2289d7d52e6da17c1e930c4ef3a4a12c639c19a9": "les modèles de scikit-learn ont-ils des arguments à régler ? Relis le §15.4.1", "b975c7515ce69ae5c66bcd651c1fc5ad80e29ee8b25e18f48bd0480b3bcca088": "une bibliothèque sait-elle quelles données tu vas lui donner ?", "ba702d2a3518d6a998ffec5f3d21734f8f404f91e2fb379e23d5cc37f24e36e1": "peut-on choisir et régler un modèle sans savoir ce qu'il fait ?"}}
📝 Ex 15.Q1b : réponse enregistrée (str).
WB_ANSWER {"hash": "7682ce6afe066fe59ceffd5767f4ef8fd58d578bd09a1e660077b50129e0206d", "id": "1

**15.Q2 — Importer proprement et lire une dépréciation**

In [4]:
wb.record("15.Q2a", "B", mistakes={"on a importé le module, pas la classe : quel préfixe faut-il écrire ?": "A",
          "le nom sklearn a-t-il été importé, et Ridge est-il rangé directement sous sklearn ?": "C",
          "linear_model est un module : peut-on l'appeler comme une fonction ?": "D"})
wb.record("15.Q2b", "B", mistakes={"un avertissement arrête-t-il le programme ?": "A",
          "pourrait-on déprécier un argument qui n'a jamais existé ?": "C",
          "relis le §15.3 : que t'annonce un avertissement de dépréciation ?": "D"})
wb.record("15.Q2c", False, mistakes={"relis le paragraphe « Suivre les dépréciations » du §15.3 : que devient un argument déprécié deux versions plus tard ?": True})
wb.record("15.Q2d", "B", mistakes={"masquer l'avertissement empêchera-t-il le code de casser à la version suivante ?": "A",
          "rester sur une vieille version, est-ce une solution durable ?": "C",
          "que se passera-t-il le jour où l'argument sera retiré ?": "D"})
wb.record("15.Q2e", False, mistakes={"relis le §15.3 : quelle version décrit l'adresse « stable » ?": True})

WB_ANSWER {"hash": "e427665d99dcdfad4cd156dee3b45517f141007f14bae07e1a8a5e547c211f35", "id": "15.Q2a", "kind": "str", "mistakes": {"45094da5e763a34261ea1195d9b60b09fdebdacccd8c9b9461435f0954513ad1": "linear_model est un module : peut-on l'appeler comme une fonction ?", "985ec0d8edbf344890fc5347648da38a2d24add2eb970f7a83eaf5bf24a5714a": "le nom sklearn a-t-il été importé, et Ridge est-il rangé directement sous sklearn ?", "a241f1c231ef30635bee1fbe6633a0ba7531d5b060e73b974398711ecd9a12e6": "on a importé le module, pas la classe : quel préfixe faut-il écrire ?"}}
📝 Ex 15.Q2a : réponse enregistrée (str).
WB_ANSWER {"hash": "95a88fcda2c5281cca97460dba0123b37b121c3b62990c0eeca93c2ca5155d3f", "id": "15.Q2b", "kind": "str", "mistakes": {"0fe4baf05ae876e521d28f7025444ae01a0a397719f3d47c1dbc0c21d46affdb": "pourrait-on déprécier un argument qui n'a jamais existé ?", "a11ca33d65973e75b8ce0666e48d7b5296b892538fa722477c3beb15d4e114d4": "relis le §15.3 : que t'annonce un avertissement de dépréciation

**15.Q3 — Hyperparamètres au constructeur, attributs appris avec « _ »**

In [5]:
wb.record("15.Q3a", "B", mistakes={"qui fixe la valeur de alpha : toi, ou l'apprentissage ?": "A",
          "une feature est une colonne des données : alpha en est-il une ?": "C",
          "un label est la réponse attendue d'un exemple : alpha en est-il un ?": "D"})
wb.record("15.Q3b", False, mistakes={"relis le §15.4.2 : que signale le tiret bas à la fin d'un nom d'attribut ?": True})
wb.record("15.Q3c", True, mistakes={"relis le §15.4.1 : que renvoie get_params ?": False})
wb.record("15.Q3d", "B", mistakes={"relis l'encadré sur clone (§15.9.3) : la copie garde-t-elle ce que l'objet a appris ?": "A",
          "clone reçoit-il des données ?": "C",
          "quel objet renvoie clone, et quelle méthode renvoie un dictionnaire ?": "D"})
wb.record("15.Q3e", False, mistakes={"relis le §15.4.1 : le constructeur reçoit-il des données ?": True})

WB_ANSWER {"hash": "08f0c6442dde3827e1526a42e9bcd98948795d2e34ff1626b8f80a949dd7f987", "id": "15.Q3a", "kind": "str", "mistakes": {"18685172a18cc2b78d00fc252dc3bfd69f3c51c46475282b6782b59e10fb7049": "qui fixe la valeur de alpha : toi, ou l'apprentissage ?", "b51eb27f96fe489668fd55f0ba84509d460b54120a96b947b44976e2e227cd24": "un label est la réponse attendue d'un exemple : alpha en est-il un ?", "ee25db3e6fe8737a1eef558d2d3108483d8dbfb3f6625e4f08b247073cbbaf54": "une feature est une colonne des données : alpha en est-il une ?"}}
📝 Ex 15.Q3a : réponse enregistrée (str).
WB_ANSWER {"hash": "e80f2d8af664bee31cfe555b512790fda30a0f2ff2dd94f14713bfa53f508bfc", "id": "15.Q3b", "kind": "bool", "mistakes": {"fe5e567130d050e34fa1d2e3662d62947ffbe8c5e9f297245e484e77efa569c3": "relis le §15.4.2 : que signale le tiret bas à la fin d'un nom d'attribut ?"}}
📝 Ex 15.Q3b : réponse enregistrée (bool).
WB_ANSWER {"hash": "d2c66d8f9b074fe2818bb3816bd55b3591a2d4b6580692c5545f0ea10a1943c0", "id": "15.Q3c", "

**15.Q4 — Ce que renvoient fit et predict**

In [6]:
wb.record("15.Q4a", "C", mistakes={"relis le §15.4.2 : comment peut-on enchaîner Ridge().fit(X, y).predict(X_new) ?": "A",
          "fit reçoit-il de nouvelles données à prédire ?": "B",
          "quelle méthode calcule le score ?": "D"})
wb.record("15.Q4b", "B", mistakes={"relis le §15.4.2 : combien de dimensions doit avoir X, même avec une seule feature ?": "A",
          "une ligne par exemple ou une ligne par feature ?": "C"})
wb.record("15.Q4c", "A", mistakes={"predict renvoie-t-il un tableau en colonnes, ou une valeur par exemple ?": "B",
          "combien d'exemples X_new contient-il, et combien de valeurs predict en renvoie-t-il ?": "C",
          "X_new contient combien d'exemples ?": "D"})
wb.record("15.Q4d", False, mistakes={"relis l'encadré « des données toujours en deux dimensions » (§15.4.3)": True})
wb.record("15.Q4e", False, mistakes={"relis le §15.4.2 : que fait un second appel à fit ?": True})

WB_ANSWER {"hash": "d0baaff77558e371bc0ca94d8b604877cb0b60f09b73dde7815d2b5eae21bf69", "id": "15.Q4a", "kind": "str", "mistakes": {"13048e3868d77c0c6183c1dda35e5a9352f48c28097ebfc33926d9a87972b39b": "relis le §15.4.2 : comment peut-on enchaîner Ridge().fit(X, y).predict(X_new) ?", "4d9107ac6634a8e36fa3d52fa7f84d32176186c3edb3bea107216a8f93286323": "fit reçoit-il de nouvelles données à prédire ?", "520f855d5918b2b94a51433c3d913758e3ef3a3a985cb0562ae007cc3ddbc832": "quelle méthode calcule le score ?"}}
📝 Ex 15.Q4a : réponse enregistrée (str).
WB_ANSWER {"hash": "db47e67a92743fd0a1ccf35affced9248a3c4dd9b5db722986bc6cb9b99aaa04", "id": "15.Q4b", "kind": "str", "mistakes": {"3843ced817e2104e644526e6c1249e8e36b40f8430fea7bec315f8f4e3f693f1": "relis le §15.4.2 : combien de dimensions doit avoir X, même avec une seule feature ?", "39830948a92b848c7e10b1b98262026414892844c4f8ea32e012783f40702603": "une ligne par exemple ou une ligne par feature ?"}}
📝 Ex 15.Q4b : réponse enregistrée (str).
WB_A

**15.Q5 — decision_function ou predict_proba ?**

In [7]:
wb.record("15.Q5a", "A", mistakes={"combien de scores un classifieur binaire renvoie-t-il par exemple ? Relis le tableau du §15.4.4": "B",
          "à deux classes, decision_function renvoie-t-il un score par classe ? Relis l'encadré ⚠️ du §15.4.4": "C",
          "une ligne par exemple, ou une ligne par classe ?": "D"})
wb.record("15.Q5b", "C", mistakes={"predict_proba donne-t-il une seule probabilité par exemple, même à deux classes ?": "A",
          "combien de colonnes faut-il pour que chaque ligne somme à 1 sur toutes les classes ?": "B",
          "une ligne par exemple, ou une ligne par classe ?": "D"})
wb.record("15.Q5c", "C", mistakes={"au-delà de deux classes, combien de scores par exemple ?": "A",
          "combien de classes a ce problème ?": "B",
          "une ligne par exemple, ou une ligne par classe ?": "D"})
wb.record("15.Q5d", float(expit(0.0)), decimals=2, mistakes={"un score nul est-il en faveur d'une classe ou à égalité ? Relis l'encadré sur la sigmoïde": 0.0,
          "que vaut e⁰ ?": 1.0})
wb.record("15.Q5e", False, mistakes={"relis le §15.4.4 : quels classifieurs n'ont pas de predict_proba ?": True})
wb.record("15.Q5f", True, mistakes={"relis le §15.4.4 : comment SVC obtient-il ses probabilités, et que dit la documentation de leur accord avec predict ?": False})

WB_ANSWER {"hash": "a32b8eef24430545306ab571a18387c738599f36ea95a693395195d468ec45ff", "id": "15.Q5a", "kind": "str", "mistakes": {"82624803b06b9d4f15014ba9d17b8c56fea62be358604aaee8b9c5074bf1112a": "une ligne par exemple, ou une ligne par classe ?", "c262c02ed8e0bbda2ba465835a5126933e08f39547ce102d83e186745862f52d": "à deux classes, decision_function renvoie-t-il un score par classe ? Relis l'encadré ⚠️ du §15.4.4", "ff0b6b77562c75d3393ddb4d7becbd7926a267d29b0daac12a2345b1df34386e": "combien de scores un classifieur binaire renvoie-t-il par exemple ? Relis le tableau du §15.4.4"}}
📝 Ex 15.Q5a : réponse enregistrée (str).
WB_ANSWER {"hash": "a69935734a6b27be609c1c2d32b8d520bdf5e527a6fa8b2d9258edb5b23d89f1", "id": "15.Q5b", "kind": "str", "mistakes": {"986b29e85efca9fe5f371d9fbceb5dee4df52a97d78ba3909e6a507f0aea0858": "predict_proba donne-t-il une seule probabilité par exemple, même à deux classes ?", "dcff40c58f12d664f96aba7e173e4f222da9064fb4e0f975b3474cb8f7f3e7a4": "une ligne par exe

**15.Q6 — KMeans : n_clusters, labels_ et choix de k**

In [8]:
wb.record("15.Q6a", KMEANS_DEFAULT_K, mistakes={"relis le §15.5 : quel est le défaut de n_clusters ?": 2,
          "relis le §15.5 : on demande le défaut de n_clusters de KMeans": 10})
wb.record("15.Q6b", False, mistakes={"relis le §15.5 : où KMeans lit-il le nombre de features ?": True})
wb.record("15.Q6c", "B", mistakes={"un clustering voit-il les vraies classes ?": "A",
          "quel attribut donne les centres ?": "C",
          "quel attribut donne l'inertie ?": "D"})
wb.record("15.Q6d", False, mistakes={"relis l'encadré sur l'inertie et la silhouette : quelle valeur de k donne la plus petite inertie possible ?": True})
wb.record("15.Q6e", "B", mistakes={"KMeans affecte-t-il chaque point à un centre, sans exception ?": "A",
          "à qui donne-t-on le nombre de groupes, dans AgglomerativeClustering(n_clusters=3) ?": "C"})

WB_ANSWER {"hash": "cc6356777e734d76639206a5c63d2403f14a5b542ac17d7621c79007403e738d", "id": "15.Q6a", "kind": "int", "magnitude": 0, "mistakes": {"00b8f64fb2225cc8d066b56c9eb34da5894939cd5f52f98dd66af12cf607204f": "relis le §15.5 : quel est le défaut de n_clusters ?", "f1128b1d6fcb56fbfcf3e1951850bef573be35cf4b1daa5fc795eb8064dbc2de": "relis le §15.5 : on demande le défaut de n_clusters de KMeans"}, "sign": 1}
📝 Ex 15.Q6a : réponse enregistrée (int).
WB_ANSWER {"hash": "f39c4fa3257e3965407b67dd62bf166af52d8396b5455eb1a4655757cd7e27a6", "id": "15.Q6b", "kind": "bool", "mistakes": {"a1e806acd0c2bfcc32efc039c356f44c777056c42af6e86dbc4021092632f77a": "relis le §15.5 : où KMeans lit-il le nombre de features ?"}}
📝 Ex 15.Q6b : réponse enregistrée (bool).
WB_ANSWER {"hash": "df333d820f750e09a3420b34f096b6cacbfd3ad9f1481206918e26145b6f5b4b", "id": "15.Q6c", "kind": "str", "mistakes": {"6ac11eab78bfd5dc7bc0dd5a36b66e366b7467b6cf7368117e7d0de4ddf34b63": "quel attribut donne l'inertie ?", "72cc3

**15.Q7 — fit, transform, fit_transform, inverse_transform**

In [9]:
wb.record("15.Q7a", "B", mistakes={"fit sur le test : quelle transformation apprendrait-on ? Relis la règle d'or du §15.6": "A",
          "fit_transform sur le test réapprend-il la transformation ? Relis le §15.6": "C",
          "inverse_transform transforme-t-il, ou revient-il en arrière ?": "D"})
wb.record("15.Q7b", (70 - 20) / (60 - 20), decimals=2, mistakes={"un scaler appris sur l'entraînement ramène-t-il toujours le test dans [0, 1] ?": 1.0,
          "x' = (x − min) / (max − min) : as-tu divisé par l'étendue ?": 50.0,
          "divise par l'étendue max − min, pas par le maximum": 0.83,
          "as-tu retiré le minimum de l'entraînement avant de diviser ?": 1.75,
          "retire le minimum, puis divise par l'étendue max − min": 1.17})
wb.record("15.Q7c", (10 - 20) / (60 - 20), decimals=2, mistakes={"un scaler appris sur l'entraînement ramène-t-il toujours le test dans [0, 1] ?": 0.0,
          "attention au signe : la valeur est sous le minimum de l'entraînement": 0.25,
          "x' = (x − min) / (max − min) : as-tu divisé par l'étendue ?": -10.0,
          "divise par l'étendue max − min, pas par le maximum": -0.17})
wb.record("15.Q7d", 20 + (60 - 20) * 0.75, decimals=1, mistakes={"x = min + (max − min) × x' : as-tu ajouté le minimum ?": 30.0,
          "multiplie par l'étendue, pas par le maximum": 65.0})
wb.record("15.Q7e", False, mistakes={"relis l'encadré ⚠️ du §15.6.1 : quels transformateurs n'ont pas d'inverse ?": True})

WB_ANSWER {"hash": "5fbbe0345e95b692c53a3ffe2514ce7ee389e0c9b06146f7a239437192c319f0", "id": "15.Q7a", "kind": "str", "mistakes": {"5ca79defbbcd687aa861ba66b677994f05ea889c90b1028d67afea025080bd53": "fit sur le test : quelle transformation apprendrait-on ? Relis la règle d'or du §15.6", "bbc5a04604eb2f837a7155ab0d978007dad9c61b46733e5c18d02825b5976574": "inverse_transform transforme-t-il, ou revient-il en arrière ?", "f5ca2fc151fae748e12c3183234048c54c108e8da034d8603e83c610108be206": "fit_transform sur le test réapprend-il la transformation ? Relis le §15.6"}}
📝 Ex 15.Q7a : réponse enregistrée (str).
WB_ANSWER {"alt_coarse": ["60093c5e81d91f4ab214dab01d2fb8bad7b511bc722f2b89f89730f37e07a0ba"], "decimals": 2, "hash": "f9f90d5d45b4c116b53d42ecaf5fe0e92c7c21b58180448e78d2611636b3ed8d", "hash_coarse": "89e3125e29c838aa850003ba927ba4d12e30f143b10cefdd090e273cd481dea7", "id": "15.Q7b", "kind": "float", "magnitude": 0, "mistakes": {"06f1796f5243370ea5a13026086ae8a3cdadaae6b8fb635dcf4aa4fba792

**15.Q8 — Ce que la PCA fait vraiment aux features**

In [10]:
wb.record("15.Q8a", "B", mistakes={"la PCA garde-t-elle des features d'origine ? Relis l'encadré ⚠️ du §15.7": "A",
          "la PCA est-elle linéaire ?": "C",
          "diviser une feature par son écart-type, est-ce une PCA ou une standardisation ?": "D"})
wb.record("15.Q8b", False, mistakes={"relis l'encadré ⚠️ du §15.7 : la PCA supprime-t-elle des features d'origine ?": True})
wb.record("15.Q8c", True, mistakes={"relis ce que fait whiten=True dans l'encadré ⚠️ du §15.7": False})
wb.record("15.Q8d", False, mistakes={"le blanchiment divise chaque composante par un nombre : change-t-il sa direction ?": True})
wb.record("15.Q8e", False, mistakes={"fit d'une PCA se sert-il de y ? Relis la PCA au ch. 12": True})

WB_ANSWER {"hash": "04752da5a1867a14f57aa38c1d89b77752960d0dc3f291b648c71be64f30ced5", "id": "15.Q8a", "kind": "str", "mistakes": {"a595ec772b40236b3c5a946c1089f26dd958eeb6256d82cbc41f58b18bd2222e": "la PCA est-elle linéaire ?", "f5f93e80c4acf8078b0c1d05c0c08ccdb51592164ef6986a7f48206f230620e1": "diviser une feature par son écart-type, est-ce une PCA ou une standardisation ?", "f84d4d9a4c97e04728b52844087239bf72d04b41b9771676cc9e735a3ac2c5b0": "la PCA garde-t-elle des features d'origine ? Relis l'encadré ⚠️ du §15.7"}}
📝 Ex 15.Q8a : réponse enregistrée (str).
WB_ANSWER {"hash": "b5fc30cb7af18bc8b845e0808c3843bc67b3d802697aedb3564328d9c83855ef", "id": "15.Q8b", "kind": "bool", "mistakes": {"d6e664b24ac02fab3f00316f7e586efe20ed55e585120b852664f5c9afd994ac": "relis l'encadré ⚠️ du §15.7 : la PCA supprime-t-elle des features d'origine ?"}}
📝 Ex 15.Q8b : réponse enregistrée (bool).
WB_ANSWER {"hash": "5cdaddf8cd6f08dcfe7db729bd9cd46dcab75567c855aa15234ed03fcba33a62", "id": "15.Q8c", "kind":

**15.Q9 — Un ensemble est un estimateur comme les autres**

In [11]:
wb.record("15.Q9a", True, mistakes={"relis le §15.8 : comment s'utilise un ensemble dans scikit-learn ?": False})
wb.record("15.Q9b", "B", mistakes={"relis l'encadré 🕰️ du §15.8 : que devient base_estimator en 1.6 ?": "A",
          "relis l'encadré 🕰️ du §15.8 : le nom de l'argument du modèle de base": "C",
          "algorithm choisissait la variante de boosting : relis l'encadré 🕰️ du §15.8": "D"})
wb.record("15.Q9c", "B", mistakes={"relis le code du §15.8 : quel est le modèle par défaut ?": "A",
          "AdaBoost de scikit-learn part-il d'un modèle linéaire par défaut ?": "C",
          "le livre passe un RidgeClassifier, mais quel est le défaut ?": "D"})
wb.record("15.Q9d", False, mistakes={"relis l'encadré ⚠️ du §15.8 : n_estimators est-il un nombre exact ?": True})
wb.record("15.Q9e", "B", mistakes={"relis l'encadré ⚠️ du §15.8 : AdaBoost donne-t-il le même poids à tous ses membres ?": "A",
          "moyenner des probabilités à poids égaux, est-ce le vote d'AdaBoost ? Relis le ch. 14": "C"})

WB_ANSWER {"hash": "2e7a17e41a876ef07e1d6ad5fe3985c1976ed48db32b28551c80d2c5163ff328", "id": "15.Q9a", "kind": "bool", "mistakes": {"84cab6730dfe86ab797df0c3e3eb37f54dbd4877308af01ec02921f67850c08f": "relis le §15.8 : comment s'utilise un ensemble dans scikit-learn ?"}}
📝 Ex 15.Q9a : réponse enregistrée (bool).
WB_ANSWER {"hash": "e6cde4bd876a9a3a5cc76699e912c3216b460f8a0d7e04cc4d10bf5bf28d7913", "id": "15.Q9b", "kind": "str", "mistakes": {"0f2410b08c9d9134134f05aaa818f1afdb0c661e91866bab69ebc1626b08d475": "relis l'encadré 🕰️ du §15.8 : le nom de l'argument du modèle de base", "8fb1e2c604487df43a56410ceb0dba7e72fea47b8329cbc48de55505109958f0": "algorithm choisissait la variante de boosting : relis l'encadré 🕰️ du §15.8", "fe6e8c3a87326350fb10431ee8c1710a2d26034d46808e3b9270a4c1d5daad95": "relis l'encadré 🕰️ du §15.8 : que devient base_estimator en 1.6 ?"}}
📝 Ex 15.Q9b : réponse enregistrée (str).
WB_ANSWER {"hash": "6fae17dc68c9a7fd7288703e29527cb547efedc400c2399a56f29c80e413997d", "id

**15.Q10 — Grille ou recherche aléatoire ?**

In [12]:
wb.record("15.Q10a", "B", mistakes={"qui choisit le nombre de clusters : toi ou l'algorithme ? Relis le §15.9": "A"})
wb.record("15.Q10b", False, mistakes={"relis le §15.9.2 : la recherche accélère-t-elle chaque entraînement ?": True})
wb.record("15.Q10c", 3 * 4 * 2, mistakes={"une grille essaie toutes les combinaisons : multiplie les nombres de valeurs, ne les additionne pas": 9})
wb.record("15.Q10d", RANDOM_DEFAULT_ITER, mistakes={"relis le §15.9.4 : la valeur par défaut de n_iter": 8,
          "relis le §15.9.4 : le défaut porte sur les combinaisons, pas sur les folds": 5})
wb.record("15.Q10e", True, mistakes={"relis l'argument de Bergstra et Bengio (§15.9.2) et la figure : combien de valeurs de h₁ essaie chaque méthode ?": False})
wb.record("15.Q10f", True, mistakes={"relis l'encadré sur les distributions (§15.9.4) : où tombent presque tous les tirages d'une uniforme entre 0,001 et 1 000 ?": False})

WB_ANSWER {"hash": "e9c6fa0d4d9cc2889b37a420f36c4186dbe83fced6ff4fd74c466e6982eab852", "id": "15.Q10a", "kind": "str", "mistakes": {"c303e36c91c141403353440b5c44c5b1a1b43ba2e7f5532eefd188c4c6ae1a91": "qui choisit le nombre de clusters : toi ou l'algorithme ? Relis le §15.9"}}
📝 Ex 15.Q10a : réponse enregistrée (str).
WB_ANSWER {"hash": "ddd39136c3224e0f4e73f161c7cadbe68069c539c34e7e38160c6dd4c5f2be22", "id": "15.Q10b", "kind": "bool", "mistakes": {"2829ff160cf4feb113eea3677b5ea421c5e1d08cc149be0d7162d4cc330dbd05": "relis le §15.9.2 : la recherche accélère-t-elle chaque entraînement ?"}}
📝 Ex 15.Q10b : réponse enregistrée (bool).
WB_ANSWER {"hash": "470b58b0f0e5256f3ce9fa76501bff88ec4c39e9c66e097a291843ee408d00a5", "id": "15.Q10c", "kind": "int", "magnitude": 1, "mistakes": {"cdec95f6a651258b3caa7e9c4b0c9d94efe1b03e2370240552ec3b5c7271f317": "une grille essaie toutes les combinaisons : multiplie les nombres de valeurs, ne les additionne pas"}, "sign": 1}
📝 Ex 15.Q10c : réponse enregistr

**15.Q11 — Pipeline : step__param et absence de fuite**

In [13]:
wb.record("15.Q11a", "D", mistakes={"le pipeline a deux étapes : comment la grille saurait-elle de quelle étape tu parles ?": "A",
          "compte les tirets bas entre le nom de l'étape et l'hyperparamètre": "B",
          "le nom de l'étape est celui que tu lui as donné dans le pipeline : relis-le": "C"})
wb.record("15.Q11b", True, mistakes={"relis le §15.9.7 : que fait cross_val_score du pipeline à chaque tour ?": False})
wb.record("15.Q11c", "B", mistakes={"un fit_transform sur le fold de validation : sur quelles données le scaler apprendrait-il ?": "A",
          "le modèle a appris sur des données standardisées : peut-il prédire sur des données brutes ?": "C",
          "un scaler appris sur toutes les lignes, c'est la situation (a) de la figure du §15.9.7": "D"})
wb.record("15.Q11d", "B", mistakes={"make_pipeline garde-t-il les majuscules du nom de la classe ? Relis le §15.9.5": "A",
          "clf est un nom choisi à la main : quel nom make_pipeline choisit-il seul ?": "C",
          "relis le §15.9.5 : d'après quoi make_pipeline nomme-t-il ses étapes ?": "D"})
wb.record("15.Q11e", False, mistakes={"relis le §15.9.5 : quel objet termine le pipeline du livre ?": True})

WB_ANSWER {"hash": "8a79e0c5b476070d4ad905d4ff31a6b552de80820dfa06898cfe43a83b756bdd", "id": "15.Q11a", "kind": "str", "mistakes": {"0bef622c6cb408c5930ba464d32d85b1d137b82cb77778b42f6cf99dd23fd2aa": "le pipeline a deux étapes : comment la grille saurait-elle de quelle étape tu parles ?", "b680a679cb001539ad951f75c0989cda5a328ff1372d7cc863c61e589ea001c9": "le nom de l'étape est celui que tu lui as donné dans le pipeline : relis-le", "e502915c88ff07210bfe9e541ce513d5bd4448afb8894535a5bb5188de885088": "compte les tirets bas entre le nom de l'étape et l'hyperparamètre"}}
📝 Ex 15.Q11a : réponse enregistrée (str).
WB_ANSWER {"hash": "48ee6940e18b8afd41ead7f84e5e08cae76ffa33202a87c9c31ea719c3af420f", "id": "15.Q11b", "kind": "bool", "mistakes": {"db18bad85cd5dd3ac99158036ee1f1eba015cd3758fcfb917db03be674ab68f8": "relis le §15.9.7 : que fait cross_val_score du pipeline à chaque tour ?"}}
📝 Ex 15.Q11b : réponse enregistrée (bool).
WB_ANSWER {"hash": "48c92c6bce7235fa7c95d2e59547f8f33dfca659271

**15.Q12 — Datasets, train_test_split et documentation**

In [14]:
wb.record("15.Q12a", len(TEST_DEFAULT) / 1000, decimals=2, mistakes={"relis le §15.11 : la part par défaut du test": 0.2,
          "c'est la part de l'entraînement : on demande celle du test": 0.75,
          "une proportion entre 0 et 1, pas un pourcentage": 25.0,
          "relis la question : on demande la valeur par défaut, quand on ne donne pas test_size": 0.3})
wb.record("15.Q12b", True, mistakes={"relis le §15.11 : que montre l'exemple des cercles générés dans l'ordre ?": False})
wb.record("15.Q12c", "B", mistakes={"ne pas mélanger garde-t-il les proportions des classes ?": "A",
          "random_state rend le découpage reproductible : garde-t-il les proportions ?": "C",
          "changer la taille du test garde-t-il les proportions ?": "D"})
wb.record("15.Q12d", False, mistakes={"relis l'encadré 🕰️ du §15.10 : que devient load_boston en 1.2 ?": True})
wb.record("15.Q12e", len(TEST_Q12), mistakes={"arrondis la taille du test au-dessus, comme scikit-learn": 37,
          "relis la question : quelle partie du découpage compte-t-on ?": 87},
          fractional="0,3 × 125 n'est pas un entier : relis la règle d'arrondi de l'énoncé")
wb.record("15.Q12f", "B", mistakes={"make_moons fait deux demi-lunes": "A", "make_blobs fait des nuages gaussiens": "C",
          "make_classification fait des problèmes réglables, pas des cercles": "D"})

WB_ANSWER {"alt_coarse": ["55990bd747ec28451c59f516777a09d46f455a452d626326288022ed67081c49"], "decimals": 2, "hash": "3ce79ac113c8647663dfca26c1eeb165ef58727e174d58f3f35f19314c16f268", "hash_coarse": "adff95fea3d40adbe67c43b47bfb4f4a2a3f7ca3cc7deb458214787b08ddf525", "id": "15.Q12a", "kind": "float", "magnitude": -1, "mistakes": {"21c515081f33c4aaaad5943964b1be4d6d20dfcaf780e5262fb98bfb981d7d10": "c'est la part de l'entraînement : on demande celle du test", "91e0bd26e0fac105fd6457a58024ff9708efad25247c4e7e0cc53de66c3c0b4b": "une proportion entre 0 et 1, pas un pourcentage", "e5385bdfe8c5963b6d1f3b7411821b784e359545d8ccb5da4eb566453edc9dec": "relis le §15.11 : la part par défaut du test", "f0486fd8129e8c29e5281fc4199a705e2c815e76c760b22e343d328d3564cde4": "relis la question : on demande la valeur par défaut, quand on ne donne pas test_size"}, "sign": 1}
📝 Ex 15.Q12a : réponse enregistrée (float, 2 décimale(s)).
WB_ANSWER {"hash": "85619bcf586c88be7c9eb9b37438404dcfde3e4a961ae3501dab9f4

**15.R1 — Forêt aléatoire ou gradient boosting ?**

In [15]:
wb.record("15.R1a", "A", mistakes={"chaque arbre d'un gradient boosting corrige-t-il les erreurs des précédents ?": "B"})
wb.record("15.R1b", False, mistakes={"relis le bilan du ch. 14 : que fait une forêt quand on lui ajoute des arbres ?": True})
wb.record("15.R1c", "A", mistakes={"le boosting combine des apprenants faibles : que leur manque-t-il, biais ou variance ?": "B"})
wb.record("15.R1d", "B", mistakes={"max_features règle le tirage des features des forêts": "A",
          "n_jobs règle le nombre de cœurs": "C", "bootstrap règle le tirage des exemples du bagging": "D"})
wb.record("15.R1e", False, mistakes={"relis le bilan du ch. 14 : quelle méthode donne un score out-of-bag ?": True})

WB_ANSWER {"hash": "8938c6fe5c67691a78f2ce81f7b0a687c2e542ad277b158c82a66f122b4cfe5b", "id": "15.R1a", "kind": "str", "mistakes": {"a4f278f6c8261d9b8685e22f2fd22f0c47b8bce50ceccacd47447cdfd902d602": "chaque arbre d'un gradient boosting corrige-t-il les erreurs des précédents ?"}}
📝 Ex 15.R1a : réponse enregistrée (str).
WB_ANSWER {"hash": "b244ef79a3e47b8aaf9e9e849097c1d0e414c41b0448685df389787d16319456", "id": "15.R1b", "kind": "bool", "mistakes": {"82f0bb4fd5ece965d5b1af58bfbf575039ac51f002fe78ba92ab2458e574eef8": "relis le bilan du ch. 14 : que fait une forêt quand on lui ajoute des arbres ?"}}
📝 Ex 15.R1b : réponse enregistrée (bool).
WB_ANSWER {"hash": "0f06de3cb42c6f401940887a7f1c27e5a2014824d8125aec683400f03d057ec2", "id": "15.R1c", "kind": "str", "mistakes": {"cdf18663add98a0dc6ba1074ed5633f8bd70f8cf855f8c7ea3fbc7b0d40ff5d6": "le boosting combine des apprenants faibles : que leur manque-t-il, biais ou variance ?"}}
📝 Ex 15.R1c : réponse enregistrée (str).
WB_ANSWER {"hash": "1b

**15.R2 — Où ajuster le scaler dans une validation croisée ?**

In [16]:
wb.record("15.R2a", True, mistakes={"relis le §12.10 : les folds de validation ont-ils servi au calcul de la moyenne et de l'écart-type ?": False})
wb.record("15.R2b", "B", mistakes={"tout le dataset contient le fold de validation : est-ce permis ?": "A",
          "apprendre le scaler sur le fold qu'on va noter, est-ce honnête ?": "C",
          "le test final sert-il pendant la validation croisée ?": "D"})
wb.record("15.R2c", (65 - 50) / 10, decimals=2, mistakes={"un z-score se calcule en retirant la moyenne : as-tu retiré 50 ?": 6.5,
          "retire la moyenne, puis divise par l'écart-type": 15.0,
          "divise par l'écart-type, pas par la variance": 0.15})
wb.record("15.R2d", True, mistakes={"relis la fin du §15.9.7 : qu'a montré l'expérience du ch. 8 sur du bruit pur ?": False})
wb.record("15.R2e", "A", mistakes={"le logarithme d'une valeur dépend-il des autres lignes ?": "B"})

WB_ANSWER {"hash": "863b05c0c95c30eb140cad0eadee7cff98367aa8bfffcae4c5ac26d9b4788892", "id": "15.R2a", "kind": "bool", "mistakes": {"2232cd57aea1842baa4374412884f91ff944fabe3c0531e4715bf88ecdf102fb": "relis le §12.10 : les folds de validation ont-ils servi au calcul de la moyenne et de l'écart-type ?"}}
📝 Ex 15.R2a : réponse enregistrée (bool).
WB_ANSWER {"hash": "e3361e6a6c2d4bb5a4d500d3289d9f7b394b5d8ac6ce3e3e66f1769ab0b62f3e", "id": "15.R2b", "kind": "str", "mistakes": {"19f37b54f60474da4f6a76b773bb5073c01df0e94e58a2f276450db163d0e16b": "tout le dataset contient le fold de validation : est-ce permis ?", "93d8297ddf974585bc1c3ae813b99774aafbdfbd5fad72c75f0366f28351b3d7": "apprendre le scaler sur le fold qu'on va noter, est-ce honnête ?", "f9e74ba431053ad557e07edbbc77f4639c4f934cd26637228d1b6b491a088b7d": "le test final sert-il pendant la validation croisée ?"}}
📝 Ex 15.R2b : réponse enregistrée (str).
WB_ANSWER {"decimals": 2, "hash": "9293a2cb53eceba7989ac21f24aae11fca0985237a04d6a6

**15.R3 — Combien de modèles pour une validation croisée à 5 folds ?**

In [17]:
wb.record("15.R3a", 5, mistakes={"relis la question : on ne compte pas de réentraînement final": 6,
          "combien de tours fait une validation croisée à 5 folds ?": 1})
wb.record("15.R3b", 1, mistakes={"chaque exemple appartient à un seul fold : combien de fois ce fold sert-il à la validation ?": 5,
          "un exemple sert à la validation dans les tours où son fold est mis de côté : combien en a-t-il ?": 4,
          "chaque exemple appartient-il à un fold ?": 0,
          "combien de fois son fold est-il mis de côté ?": 2})
wb.record("15.R3c", 4, mistakes={"un exemple ne sert pas à l'entraînement quand son fold est mis de côté": 5,
          "relis la question : on demande combien de fois il sert à l'entraînement": 1})
wb.record("15.R3d", LARGEST_FOLD_R3, mistakes={"1 003 n'est pas divisible par 5 : que fait KFold du reste ?": 200,
          "c'est la taille de l'entraînement d'un tour, pas d'un fold": 802,
          "KFold met-il tout le reste dans un seul fold ?": 203},
          fractional="un fold a un nombre entier d'exemples : relis comment KFold répartit le reste")
wb.record("15.R3e", 40, mistakes={"un leave-one-out met-il de côté chaque exemple, une fois ?": 39,
          "combien de tours fait un leave-one-out ?": 1})
wb.record("15.R3f", True, mistakes={"relis le §15.9.1 : à quoi sert la stratification ?": False})

WB_ANSWER {"hash": "30120a630dfd90c7090ab57e92d8c64284d8960c5084e720c60029f0aee91bde", "id": "15.R3a", "kind": "int", "magnitude": 0, "mistakes": {"055cdc96f6a0fb014c232959de64b6f6e1b6bc22fe2a69291c722fdb688e627d": "relis la question : on ne compte pas de réentraînement final", "72c5350f16e16fb8fe39e45d9c8fe9fb622deec689e2e4a9bcef8bcafed6b90a": "combien de tours fait une validation croisée à 5 folds ?"}, "sign": 1}
📝 Ex 15.R3a : réponse enregistrée (int).
WB_ANSWER {"hash": "f2cc984506af922e6da1292c919bfe756bdeebedb2acb2fea0e8e02616e13ba7", "id": "15.R3b", "kind": "int", "magnitude": 0, "mistakes": {"2b4eaee76b0ba0d1f695ea317745fa8eec589ae2f770989043c403e6746f57e3": "combien de fois son fold est-il mis de côté ?", "46776091c0ebc2ed94746859f9583362a611cd24d92f11a699e287b4ecfdd563": "un exemple sert à la validation dans les tours où son fold est mis de côté : combien en a-t-il ?", "d9c9b6b9d7707893d9a9d96cca482a80e40dc1427d613d6b5cfcf6ac6608ac76": "chaque exemple appartient-il à un fold 

**Ex 15.1 — Compter les modèles entraînés par une recherche en grille**

In [18]:
wb.record("15.1a", N_CAND_151, mistakes={"une grille essaie toutes les combinaisons : multiplie les nombres de valeurs, ne les additionne pas": 7})
wb.record("15.1b", fits(N_CAND_151, 5, refit=False), mistakes={"chaque candidat est noté par une validation croisée à 5 folds : combien d'entraînements par candidat ?": 12,
          "relis la question : on demande les entraînements des validations croisées, sans le réentraînement final": 61})
wb.record("15.1c", fits(N_CAND_151, 5), mistakes={"as-tu compté le réentraînement final du meilleur réglage (refit=True) ?": 60,
          "un seul réentraînement final, celui du meilleur candidat, pas un par candidat": 72,
          "chaque candidat est noté par une validation croisée à 5 folds": 13})
wb.record("15.1d", fits(N_CAND_151, 3), mistakes={"as-tu compté le réentraînement final ?": 36,
          "relis la question : avec combien de folds ?": 61})
wb.record("15.1e", fits(POLY_151, 4), mistakes={"as-tu compté le réentraînement final ?": 120,
          "interaction_only est aussi un hyperparamètre de la grille : combien de valeurs a-t-il ?": 61,
          "relis cv : combien de folds ?": 151,
          "on compte les entraînements du pipeline entier, pas de chacune de ses étapes": 242})
wb.record("15.1f", fits(LIST_151, 5, refit=False), mistakes={"refit=False : pas de réentraînement final": 46,
          "les combinaisons des deux dictionnaires s'additionnent, elles ne se multiplient pas": 90,
          "chaque dictionnaire est une grille à part : linear n'a pas de gamma": 60,
          "as-tu compté les deux valeurs de gamma dans la seconde grille ?": 30})
wb.record("15.1g", fits(20, 5), mistakes={"as-tu compté le réentraînement final ?": 100, "chaque combinaison est-elle notée une seule fois ?": 21})
wb.record("15.1h", 3 * fits(N_CAND_151, 5), mistakes={"chaque fold extérieur relance une recherche complète, réentraînement final compris": 180,
          "combien de recherches complètes la validation croisée extérieure lance-t-elle ?": 61,
          "cross_val_score n'ajoute pas de réentraînement final après ses 3 folds": 184})

WB_ANSWER {"hash": "510a8cf1d2938c232d79961186ebaaa92335669674733113c99fe8710725af53", "id": "15.1a", "kind": "int", "magnitude": 1, "mistakes": {"583722b249a1b51d635f4709669fa4bffcf60d0b73145195a018c35286297481": "une grille essaie toutes les combinaisons : multiplie les nombres de valeurs, ne les additionne pas"}, "sign": 1}
📝 Ex 15.1a : réponse enregistrée (int).
WB_ANSWER {"hash": "c3e2acd0489bb2164c7037a8a2c011128fd2ad1a0d0644488d9c8839c107cd86", "id": "15.1b", "kind": "int", "magnitude": 1, "mistakes": {"7d1194d9d5cb7c24de758decd097e8b6651eb0b2f6eee9325d5e5d2cd836b397": "relis la question : on demande les entraînements des validations croisées, sans le réentraînement final", "8d68bf7f23c430b34e7552f1c29bf5ef0ee9657f357c2db8a9115061e9680770": "chaque candidat est noté par une validation croisée à 5 folds : combien d'entraînements par candidat ?"}, "sign": 1}
📝 Ex 15.1b : réponse enregistrée (int).
WB_ANSWER {"hash": "64d2eae122d0283363d460914c71a92b9d9a63e0e2d32e8e273a727f74069279

**Ex 15.2 — De decision_function à predict_proba**

In [19]:
wb.record("15.2a", float(P_152[0]), decimals=4, mistakes={"σ(s) = 1 / (1 + e^(−s)) : attention au signe de l'exposant": 0.1192,
          "une probabilité peut-elle dépasser 1 ?": 7.3891})
wb.record("15.2b", float(P_152[1]), decimals=4, mistakes={"σ(s) = 1 / (1 + e^(−s)) : attention au signe de l'exposant": 0.7311,
          "e^s seul n'est pas la sigmoïde : relis sa formule": 0.3679})
wb.record("15.2c", float(P_152[2]), decimals=2, mistakes={"que vaut e⁰ ?": 1.0, "un score nul penche-t-il d'un côté ?": 0.0})
wb.record("15.2d", PRED_152, mistakes={"relis le §15.4.4 : la classe 1 exige-t-elle un score positif ou nul, ou strictement positif ?": [1, 0, 1, 1, 0],
          "le signe du score donne la classe : la classe 1 pour un score positif": [0, 1, 1, 0, 1],
          "le seuil de predict porte sur le score : quel score correspond à la probabilité 0,5 ?": [1, 0, 0, 0, 0]})
wb.record("15.2e", logit(0.9), decimals=4, mistakes={"le logit est ln(p / (1 − p)), pas ln(p)": -0.1054,
          "le logit utilise le logarithme népérien, pas log₁₀": 0.9542,
          "c'est ln(p / (1 − p)) : as-tu inversé la fraction ?": -2.1972})
wb.record("15.2f", logit(0.8), decimals=4, mistakes={"le logit est ln(p / (1 − p)), pas ln(p)": -0.2231,
          "le logit utilise le logarithme népérien, pas log₁₀": 0.6021,
          "c'est ln(p / (1 − p)) : as-tu inversé la fraction ?": -1.3863})
wb.record("15.2g", int(np.sum(S_152 >= logit(0.8))), mistakes={"compare chaque score au seuil de f), pas à 0": 2, "au moins un score dépasse le seuil de f) : lequel ?": 0})
wb.record("15.2h", float(SOFT_152[0]), decimals=4, mistakes={"le softmax divise e^(s₀) par la somme des e^(sⱼ) de TOUTES les classes": 0.7311,
          "le softmax travaille sur les exponentielles, pas sur les scores eux-mêmes": 1.0,
          "le softmax travaille sur les exponentielles de toutes les classes : as-tu divisé les scores bruts des deux premières ?": 0.6667,
          "as-tu gardé le signe du troisième score (−1) dans son exponentielle ?": 0.5761})
wb.record("15.2i", False, mistakes={"relis l'encadré sur le softmax : que devient e^(s+5) / Σ e^(sⱼ+5) ?": True})
wb.record("15.2j", False, mistakes={"relis le §15.4.4 : pourquoi RidgeClassifier n'a-t-il pas de predict_proba ?": True})

WB_ANSWER {"decimals": 4, "hash": "71f0181e6011ca60f4c8a0869f397b91654062284b5504ee5be8e052b22bdc58", "hash_coarse": "d937e11588d5352d54c89242ac9af8a508e76c23f168cf14d5a604ebb7182495", "id": "15.2a", "kind": "float", "magnitude": -1, "mistakes": {"97337258da57226284d380ddee674fbe474be5d54401f33868aa577235148a55": "une probabilité peut-elle dépasser 1 ?", "a27b16488cdc664bf4f40a919d956061fe5d700dfe54b8f077cfd126345788db": "σ(s) = 1 / (1 + e^(−s)) : attention au signe de l'exposant"}, "sign": 1}
📝 Ex 15.2a : réponse enregistrée (float, 4 décimale(s)).
WB_ANSWER {"decimals": 4, "hash": "628a86799fecb2758afa531b1d2e7ea31e622cb1204331a22a8f86caaeac954d", "hash_coarse": "6ddddaf0dc3b59642373d40d0d1cb562a58e3ca1cca6cda65fc6b79309e39258", "id": "15.2b", "kind": "float", "magnitude": -1, "mistakes": {"1de9e6cff6f5bb199d05b239618afb7c12fc74920d2fd93cd0fc3bf78bc1d363": "e^s seul n'est pas la sigmoïde : relis sa formule", "4477c5fcacf1cd514c0a9cca07330cf5bc2da05124f036c1fbed356391d635ac": "σ(s) = 

**Ex 15.3 — Ramener une droite ajustée dans l'espace d'origine**

In [20]:
wb.record("15.3a", x_scaled(1.0), decimals=3, mistakes={"x' = (x − min) / (max − min) : as-tu retiré le minimum, −3 ?": 0.125,
          "divise par l'étendue max − min, pas par le maximum": 0.8,
          "retirer un minimum négatif, c'est ajouter : attention au signe de −3": -0.25})
wb.record("15.3b", A_153 * x_scaled(1.0) + B_153, decimals=3, mistakes={"la droite s'applique à x', pas à x : as-tu d'abord transformé x ?": 0.85,
          "as-tu ajouté l'ordonnée à l'origine 0,25 ?": 0.3})
wb.record("15.3c", line_153(1.0), decimals=2, mistakes={"y = min + (max − min) × y' : as-tu ajouté le minimum de y ?": 11.0,
          "ramène y' avec l'étendue de y, pas celle de x": 14.4,
          "la droite s'applique à x', pas à x : as-tu d'abord transformé x ?": 27.0,
          "relis la question : on demande la prédiction dans l'unité d'origine de y": 0.55})
wb.record("15.3d", line_153(X_MIN), decimals=2, mistakes={"y = min + (max − min) × y' : as-tu ajouté le minimum de y ?": 5.0,
          "en x' = 0, y' vaut l'ordonnée à l'origine de la droite transformée : as-tu pris 0 ?": 10.0,
          "c'est encore une valeur transformée : ramène-la dans l'unité de y": 0.25})
wb.record("15.3e", line_153(X_MAX), decimals=2, mistakes={"y = min + (max − min) × y' : as-tu ajouté le minimum de y ?": 17.0,
          "en x' = 1, y' = 0,6 + 0,25 : as-tu oublié l'ordonnée à l'origine ?": 22.0,
          "c'est encore une valeur transformée : ramène-la dans l'unité de y": 0.85})
wb.record("15.3f", SLOPE_153, decimals=3, mistakes={"la pente transformée est-elle dans les unités d'origine ?": 0.6,
          "une pente se mesure en unités de y par unité de x : as-tu inversé le rapport ?": 0.24,
          "as-tu divisé par l'étendue de x ?": 12.0,
          "as-tu oublié la pente 0,6 de la droite transformée ?": 2.5})
wb.record("15.3g", line_153(0.0), decimals=2, mistakes={"x = 0 n'est pas x' = 0 : transforme d'abord x = 0": 15.0,
          "ramène y' dans l'unité d'origine avec y = min + (max − min) × y'": 0.475,
          "c'est l'ordonnée à l'origine de la droite transformée : ramène-la dans l'espace d'origine, en x = 0": 0.25})
wb.record("15.3h", x_scaled(11.0), decimals=3, mistakes={"x' = (x − min) / (max − min) : as-tu retiré le minimum, −3 ?": 1.375,
          "un point hors de l'étendue de l'entraînement sort de [0, 1] : faut-il le ramener à 1 ?": 1.0})
wb.record("15.3i", line_153(11.0), decimals=2, mistakes={"un point hors de l'étendue sort de [0, 1] : faut-il le ramener à 1 avant la droite ?": 27.0,
          "y = min + (max − min) × y' : as-tu ajouté le minimum de y ?": 26.0})
wb.record("15.3j", False, mistakes={"fais-le : que donne inverse_transform sur le point (0,6 ; 0,25), et est-ce la pente trouvée en f) ?": True})

WB_ANSWER {"decimals": 3, "hash": "1c51bd233a6e3656233628fee3e1d887680baca7172cf26b8a194a701634d2dc", "hash_coarse": "bb1b5c6a33047726d3afb1be04b4a4c9ffdec9a9f275aac42da770c980aa039c", "id": "15.3a", "kind": "float", "magnitude": -1, "mistakes": {"162d65c4bca1b2a4a33f5270a0b2fbaad6e9f6b06cac1b5a96648a8a03d35f0c": "divise par l'étendue max − min, pas par le maximum", "7a6ed4695feb24861bd4cbe4844cb39830d956ac4f09c1d1468067130560ac38": "x' = (x − min) / (max − min) : as-tu retiré le minimum, −3 ?", "f26d964040553a62b82c04f1ca5314e04031776b8bab2b253fb9a5f5c1d57a6c": "retirer un minimum négatif, c'est ajouter : attention au signe de −3"}, "sign": 1}
📝 Ex 15.3a : réponse enregistrée (float, 3 décimale(s)).
WB_ANSWER {"decimals": 3, "hash": "612f86ac8037f70d7d92919635e60346c4c9dc72ea1f0551f7fd0b4de626bcbe", "hash_coarse": "89366f82994b30adc16d7d8e49495bbf508d954d5e0d80e74995cb6739a62567", "id": "15.3b", "kind": "float", "magnitude": -1, "mistakes": {"7d7cfb5d7e36123df6c72b8d61a0f936860bc6d4c7

**Ex 15.4 — Combien de features polynomiales ?**

In [21]:
wb.record("15.4a", n_poly(2, 2), mistakes={"as-tu compté la constante 1 (include_bias=True) ?": 5,
          "x·y et y·x sont le même monôme : l'ordre des facteurs ne compte pas": 7,
          "compte aussi les monômes de degré 0 et 1, pas seulement ceux de degré 2": 3})
wb.record("15.4b", n_poly(2, 5), mistakes={"as-tu compté la constante 1 ?": 20,
          "l'ordre des facteurs ne compte pas : ce n'est pas un nombre de suites": 63,
          "compte tous les degrés de 0 à 5, pas seulement le degré 5": 6})
wb.record("15.4c", comb(2 + 4 - 1, 4), mistakes={"on demande le degré EXACTEMENT 4 : retire les monômes de degré plus petit": 15,
          "énumère-les : x⁴, x³y, …": 4,
          "l'ordre des facteurs ne compte pas : x³y et yx³ sont le même monôme": 16})
wb.record("15.4d", n_poly(8, 2), mistakes={"as-tu compté la constante 1 ?": 44, "compte aussi les carrés x₁², …, x₈²": 37,
          "l'ordre des facteurs ne compte pas : x₁x₂ et x₂x₁ sont le même monôme": 73})
wb.record("15.4e", n_poly(8, 3), mistakes={"as-tu compté la constante 1 ?": 164,
          "dans C(p + d, d), le nombre du haut est p + d : relis l'encadré": 120})
wb.record("15.4f", n_poly(20, 3, bias=False), mistakes={"include_bias=False retire la constante": 1771,
          "dans C(p + d, d), le nombre du haut est p + d : relis l'encadré": 1539})
wb.record("15.4g", 1 + 5 + comb(5, 2), mistakes={"interaction_only=True retire les carrés comme x₁², mais garde la constante et les x_j": 21,
          "as-tu compté la constante 1 ?": 15, "as-tu gardé les cinq features de degré 1 ?": 11})
wb.record("15.4h", 100_000 * n_poly(20, 3) * 8 / 1e9, decimals=2, mistakes={"compte 8 octets par nombre": 0.18,
          "l'énoncé compte en Go de 10⁹ octets, pas de 2³⁰": 1.32,
          "un Go vaut 10⁹ octets : convertis les octets en Go": 1416800000.0})
wb.record("15.4i", SMALLEST_D_154, mistakes={"as-tu vérifié que C(10 + d, d) dépasse 1 000 pour ta valeur de d ?": 3,
          "teste les degrés un par un, en partant de 1": 5})

WB_ANSWER {"hash": "f4d80f531b012c2344dc5c5e5c93d008b16af23fd218209303d1dbcde8cf0414", "id": "15.4a", "kind": "int", "magnitude": 0, "mistakes": {"479fa5d9737e7081389b4ebd57bd073d95a6f7101323b2079bddd1842e394d5a": "as-tu compté la constante 1 (include_bias=True) ?", "722a49d28e99de82488ceaa375329abffe5b6ff6e1f3805d27e0425a29d58900": "compte aussi les monômes de degré 0 et 1, pas seulement ceux de degré 2", "f01e5fc940e6a7d1ffb56dbafc7d04b2d8e52cba388dcc5ec88b510b198925dc": "x·y et y·x sont le même monôme : l'ordre des facteurs ne compte pas"}, "sign": 1}
📝 Ex 15.4a : réponse enregistrée (int).
WB_ANSWER {"hash": "a0583731e9ceaa9fa931c6b1571445deb7103f2437be06b97345b16cd1d18e8d", "id": "15.4b", "kind": "int", "magnitude": 1, "mistakes": {"24873d0a02a1bff5e8ddc315ef4706da50bf920a850a3945e66f61ab22c14101": "as-tu compté la constante 1 ?", "2b6da770eba05294c84ce2e36464f0ed139516b6ae855adcb83e6265918ef2d4": "l'ordre des facteurs ne compte pas : ce n'est pas un nombre de suites", "4ffc073e62

**Ex 15.5 — Budget d'une recherche d'hyperparamètres**

In [22]:
wb.record("15.5a", FITS_155, mistakes={"chaque combinaison est notée sur 5 folds": 3125,
          "les nombres de valeurs des 5 hyperparamètres se multiplient entre eux": 125})
wb.record("15.5b", DAYS_155, decimals=1, mistakes={"c'est une durée en minutes : convertis-la en jours": 62500.0,
          "c'est une durée en heures : convertis-la en jours": 1041.7,
          "as-tu compté les 5 folds ?": 8.7,
          "62 500 est en minutes : par quoi as-tu divisé pour obtenir des jours ?": 2604.2})
wb.record("15.5c", DAYS_155 / 8, decimals=1, mistakes={"le parallélisme divise la durée, il ne la multiplie pas": 347.2,
          "combien de cœurs travaillent en même temps ?": 21.7})
wb.record("15.5d", 60 * 5 * 4 / 8 / 60, decimals=1, mistakes={"as-tu divisé par les 8 cœurs ?": 20.0, "as-tu compté les 5 folds ?": 0.5,
          "convertis les minutes en heures": 150.0})
wb.record("15.5e", 1 - 0.95 ** 60, decimals=4, mistakes={"c'est la probabilité qu'AUCUN essai ne tombe dans les 5 % : on demande le complément": 0.0461,
          "les essais ne s'additionnent pas : 60 × 0,05 dépasse 1": 3.0,
          "quelle est la probabilité qu'un seul essai RATE la bonne zone ?": 1.0})
wb.record("15.5f", N_99_155, mistakes={"le nombre d'essais est un entier : arrondis au-dessus pour atteindre au moins 0,99": 89,
          "dans le dénominateur, quelle probabilité as-tu mise : celle de réussir ou celle de rater un essai ?": 2,
          "c'est le nombre pour 0,95, pas pour 0,99": 59},
          fractional="un nombre d'essais est entier : faut-il arrondir au plus proche, ou au-dessus pour atteindre 0,99 ?")
wb.record("15.5g", HALVING_155, decimals=1, mistakes={"as-tu compté tous les tours ?": 4.0,
          "additionne le coût de TOUS les tours, pas seulement le dernier": 1.0,
          "chaque candidat d'un tour ne reçoit qu'une part des données : en as-tu tenu compte ?": 121.0})
wb.record("15.5h", 81 / HALVING_155, decimals=1, mistakes={"on demande le coût de la grille divisé par celui du successive halving, pas l'inverse": 0.1,
          "relis g) : le successive halving coûte-t-il un seul entraînement ?": 81.0,
          "as-tu repris le coût de g) avec ses 5 tours ?": 20.3,
          "relis g) : combien de tours as-tu comptés ?": 20.2})

WB_ANSWER {"hash": "e91d67f4a5802196ff848d79956fc418f9eb17a11a8ffa37f611c2f06b190b3b", "id": "15.5a", "kind": "int", "magnitude": 4, "mistakes": {"5f25a1e5a71e9390df8004da810f59fe85beab7b3b534104a0d522428b646eef": "les nombres de valeurs des 5 hyperparamètres se multiplient entre eux", "a1b14137abbe579433625020d48e4be37c2c8722e792f4f7a7020b4630dbc93d": "chaque combinaison est notée sur 5 folds"}, "sign": 1}
📝 Ex 15.5a : réponse enregistrée (int).
WB_ANSWER {"decimals": 1, "hash": "46ac4ca0bee47d885b9ecefa25a75db7bc8bdab821d1945d0d143fe03ded1d62", "hash_coarse": "ecf3b19328319c4447aeb212393e2d56a776d635935111384b0b90089c8ec49e", "id": "15.5b", "kind": "float", "magnitude": 1, "mistakes": {"2b77767f8fbed91c6e57bd93b1b15e4d0378af4ae78cd91b39ce58a3a5c5d4ab": "as-tu compté les 5 folds ?", "703f63062ab5118fced92aa705148f042e3db5ccb68976e82c4559f03d2e8dcd": "c'est une durée en minutes : convertis-la en jours", "e9784738b188620f35926a00efe69fca044f30dc9527542e31b80fbe88414cad": "c'est une duré

## ✅ Bilan

**Auto-évaluation** (note-toi de 0 à 3 dans `mon_travail/suivi/auto_evaluation.md`) :
1. Sais-tu dire ce que renvoient `fit`, `predict`, `decision_function`, `predict_proba` et `transform`, et sous quelle forme ?
2. Sais-tu construire un pipeline qui se règle par une recherche en grille ou au hasard sans fuite, et estimer le coût de cette recherche ?
3. Sais-tu moderniser un appel déprécié en lisant la documentation de ta version, et sauvegarder un modèle avec ses versions ?

**Pour aller plus loin** : les articles de Pedregosa et coll. (2011), de Buitinck et coll. (2013) et de Bergstra et Bengio (2012), la page « Common pitfalls » de la documentation, cités dans la fiche. La suite : le checkpoint de la partie III (ML classique), puis les réseaux de neurones (ch. 16).

*Les exercices de code 15.9 à 15.31 (parties A à D) seront ajoutés à la prochaine session de génération ; relance alors `python tools/start_chapter.py 15` pour obtenir le notebook complet.*